# Qiskit Fall Fest 2026, Challenge 4: Quantum kernel classifier

Opens Thu, Nov 19, 2026 at 6:00 PM. Closes Sun, Nov 22, 2026 at 11:59 PM. Both times are Clemson time (US Eastern).

**The task.** Reach the highest accuracy on a held-out test set.

Start from Thursday's notebook. Its Lab 2 built a kernel matrix, a table that says how alike every two data points are, and its Lab 3 trained a classifier, a program that sorts points into two classes, on that table and scored it on test points. This notebook gives you a new data set: 60 training points with their labels (the class of each point, 0 or 1), and 100 test points without. You train on the first and predict the labels of the second. Held-out means that the organizers keep the labels of the test points.

How to use this notebook:

- Run the three code cells of the Setup section first. This challenge needs no quantum computer and no account, so leave the three switches as they are.
- Read "The task in detail". It says what the data is, what you are told about it, and how your answer is scored.
- Run the cells under "The tools". Their code is complete. They hold the data, Thursday's code, a way to judge a setting without the test labels, and the check that writes your line.
- Do your own work under "Your attempt".
- Turn in the line that the notebook prints. The steps are under "Turn it in". Only three lines per person count, so read that part before you send anything.

## Setup

Run the next three code cells in order, every time you open this notebook.

**Where your circuits run.** The first cell holds three switches.

- `USE_HARDWARE = False` keeps everything on the computer that runs this notebook (your laptop, or the server of Google Colab or qBraid Lab, two websites that run notebooks in your browser). Circuits run on a simulator, a program that imitates a quantum computer. No account is needed. Use this setting during the lab.
- `USE_HARDWARE = True` with `HARDWARE = "ibm"` sends the hardware cells to a real IBM quantum processor through your own IBM Quantum account. A hardware cell is a cell that can send work to a real quantum computer, and a processor is the chip that holds its qubits. This is the first choice for real hardware.
- `USE_HARDWARE = True` with `HARDWARE = "qbraid"` is the second option. qBraid is a service that gives access to quantum computers from several makers. The cells marked "second option" then use your qBraid account to reach the other quantum computers qBraid offers. qBraid calls each of them a device, and `QBRAID_DEVICE` holds the id of the device you pick there.

After you change a switch, run the switches cell below and the setup cell after it again.

In [ ]:
USE_HARDWARE = False      # True sends jobs to a real quantum computer
HARDWARE = "ibm"          # "ibm" (first choice) or "qbraid" (second option)
QBRAID_DEVICE = "PASTE_DEVICE_ID"

The next cell checks that the tested versions of the packages are installed. If one is missing or has a different version, the cell installs it and then stops with one line that asks you to restart the kernel. The kernel is the Python process that runs the cells of this notebook. After the restart, run the notebook again from the top.

The cell also defines `connect()`, the one helper that every hardware cell calls. `connect("ibm")` gives the lab code a service, the object you ask for a backend. A backend is the machine, real or simulated, that runs a circuit. With `USE_HARDWARE = False` the service hands out a simulated copy of the IBM processor `ibm_fez`, built from a saved record of that machine's error rates. Qiskit names this copy `fake_fez`.

In [ ]:
# Setup. Run this cell every time, right after the switches cell above. It checks the packages, then defines connect().
import sys
from importlib.metadata import PackageNotFoundError, version

PINNED = {"qiskit": "2.6.2", "qiskit-ibm-runtime": "0.52.1", "qiskit-aer": "0.18.2"}
if str(HARDWARE).strip().lower() == "qbraid":
    PINNED["qbraid"] = "0.13.0"                        # needed only for the second option
EXTRA = ["matplotlib", "pylatexenc", "scikit-learn"]   # any version will do


def _found(name):
    try:
        return version(name)
    except PackageNotFoundError:
        return None


def _missing():
    wanted = [f"{name}=={pin}" for name, pin in PINNED.items() if _found(name) != pin]
    return wanted + [name for name in EXTRA if _found(name) is None]


class StopHere(Exception):
    """Stops the run with one plain line in place of a long error report."""

    def _render_traceback_(self):
        return [str(self)]


if sys.version_info < (3, 11):
    raise StopHere("This notebook needs Python 3.11 or newer. This kernel runs Python "
                   + sys.version.split()[0] + ". Choose a newer Python, then run this cell again.")

if _missing():
    _spec = " ".join(_missing())
    %pip install -q {_spec}
    if _missing():
        raise StopHere("Some packages could not be installed: " + ", ".join(_missing())
                       + ". Read the message above this line, or ask an organizer.")
    raise StopHere("Packages were installed. Restart the kernel now, then run the notebook "
                   "again from the top. (Jupyter and qBraid Lab: Kernel menu, Restart Kernel. "
                   "Google Colab: Runtime menu, Restart session.)")

import os
from getpass import getpass

from qiskit_ibm_runtime import QiskitRuntimeService
from qiskit_ibm_runtime.accounts import AccountNotFoundError
from qiskit_ibm_runtime.fake_provider import FakeFez

class SimulatedBackend(FakeFez):
    """A simulated copy of the IBM processor ibm_fez. It runs where this notebook runs, and nothing is sent."""

    def __repr__(self):
        return f"<{type(self).__name__}('{self.name}')>"


SIMULATED_BACKEND = SimulatedBackend()


class SimulatedService:
    """Stands in for QiskitRuntimeService when nothing is sent to IBM.

    It answers the same calls and always hands out the simulated processor.
    """

    def backends(self, **filters):
        return [SIMULATED_BACKEND]

    def backend(self, name=None, **filters):
        return SIMULATED_BACKEND

    def least_busy(self, **filters):
        return SIMULATED_BACKEND


def connect(account):
    """Return the entry point for one account. Every hardware cell calls this.

    connect("ibm") returns a service. service.least_busy(...) then gives a backend.
        USE_HARDWARE = True and HARDWARE = "ibm": your own IBM Quantum account.
        Any other setting: a simulated IBM processor. No account is used.
    connect("qbraid") returns a qBraid provider, or None when that route is off.
        USE_HARDWARE = True and HARDWARE = "qbraid": your qBraid account.
        Any other setting: None. Nothing is sent.
    """
    route = str(HARDWARE).strip().lower()
    if not isinstance(USE_HARDWARE, bool) or route not in ("ibm", "qbraid"):
        raise ValueError("Check the first code cell: USE_HARDWARE must be True or False, "
                         'and HARDWARE must be "ibm" or "qbraid".')
    if account not in ("ibm", "qbraid"):
        raise ValueError('Call connect("ibm") or connect("qbraid").')
    live = USE_HARDWARE and route == account
    why = "USE_HARDWARE is False" if not USE_HARDWARE else f'HARDWARE is "{route}"'

    if account == "ibm":
        if not live:
            print(f"Simulator: this cell runs on {SIMULATED_BACKEND.name}, a simulated IBM "
                  f"processor ({why}). No account is used and nothing is sent.")
            return SimulatedService()
        try:
            service = QiskitRuntimeService()
        except AccountNotFoundError:
            print("No IBM Quantum account is saved on this computer. Paste your API key and "
                  "your instance CRN in the boxes. They stay hidden. They are saved in your "
                  "home folder, never in this notebook.")
            QiskitRuntimeService.save_account(
                token=getpass("IBM Quantum API key: "),
                instance=getpass("Instance CRN (press Enter to skip): ") or None,
                set_as_default=True, overwrite=True)
            service = QiskitRuntimeService()
        print("Real hardware: this cell uses your own IBM Quantum account and your quantum time.")
        return service

    if not live:
        print(f"Skipped: this cell uses a qBraid account, the second option ({why}). "
              "Nothing is sent.")
        return None
    from qbraid.runtime import QbraidProvider
    if not (os.environ.get("QBRAID_API_KEY")
            or os.path.exists(os.path.expanduser("~/.qbraid/qbraidrc"))):
        print("No qBraid API key is saved on this computer. Paste it in the box. It stays "
              "hidden and is kept only until this kernel stops.")
        os.environ["QBRAID_API_KEY"] = getpass("qBraid API key: ")
    if QBRAID_DEVICE == "PASTE_DEVICE_ID":
        print('QBRAID_DEVICE is still "PASTE_DEVICE_ID". Copy the id of an online device from '
              "the list this cell prints into the first code cell, run that cell, then run "
              "this cell again.")
    print("Real hardware, second option: this cell uses your qBraid account. Device:", QBRAID_DEVICE)
    return QbraidProvider()

The setup check prints what is installed and where circuits will run. If the versions match the tested ones, you are ready.

In [ ]:
# Setup check: what is installed, and where circuits will run.
import qiskit, qiskit_aer, qiskit_ibm_runtime   # an error here means an install went wrong

print("Python", sys.version.split()[0])
for _name, _pin in PINNED.items():
    print(f"{_name} {_found(_name)}  (tested with {_pin})")
print("USE_HARDWARE =", USE_HARDWARE, "| HARDWARE =", repr(HARDWARE),
      "| QBRAID_DEVICE =", repr(QBRAID_DEVICE))
print("Simulated IBM processor:", SIMULATED_BACKEND.name, "with",
      SIMULATED_BACKEND.num_qubits, "qubits")

## The task in detail

### The data

Every point has two features, and each feature is a number between 0 and 1. A feature is one of the numbers that describe a data point. Every point also has a label, 0 or 1, and the label is what a classifier predicts.

- **Training set.** 60 points with their labels, 30 of each class.
- **Test set.** 100 points. Their labels are not in this notebook.

This challenge has a data set of its own. It is not Thursday's, so Thursday's accuracy of 0.8667 does not carry over.

### What you are told about the data

- **The labels.** They were made with Thursday's feature map, `zz_feature_map(2, reps=2)`. A feature map is the circuit that turns a data point into a quantum state. Because the labels come from this very map, a full score is possible.
- **From features to angles.** The features are plain numbers between 0 and 1, and a feature map needs angles. Here each feature became an angle by multiplying it by one number. The number is the same for both features, and nothing was added.
- **The number.** It lies between 1 and 10 and has two decimals. It is not Thursday's $\frac{\pi}{2}$.
- **The test labels.** They are hidden. So judge a setting on the training set alone.

### The rules

- The challenge is scored individually. Your labels have to be your own work. Do not post or send your line or your labels to anyone before the close, in Discord or anywhere else.
- No quantum computer is needed. As in Thursday's Lab 2, every entry of the table is computed on the computer that runs this notebook, from the exact states of two points. A real processor would need one circuit for every pair of points: 1,770 for the training set and 6,000 more for the test set.
- You turn in one predicted label, 0 or 1, for each of the 100 test points, in their order.
- Only your first three well-formed lines count, and the best of them is your score. A fourth line is ignored. A line is well-formed when it has the form of the line this notebook prints: `QFF26-C4 labels=` followed by 100 characters, each of them 0 or 1.

### Your score

Guessing gets 50 of the 100 test labels right on average. So the score counts what you get right beyond 50:

$$\text{score} = \max\left(0, \; 2 \times (\text{correct} - 50)\right)$$

Here "correct" is the number of test points with the right label. 75 right labels score 50, and 100 score 100. How the event uses the scores is explained on the event page.

### What to expect

The notebook as given uses Thursday's $\frac{\pi}{2}$. With it the classifier gets 50 of the 100 test labels right, which is a score of 0. You do not need to spend one of your three lines to find that out. The work is to find the number. With the right one, a full score is within reach.

## The tools

The code in this part is complete. Run each cell and read what it does.

The first cell holds the data: `U_train` with the features of the 60 training points, `y_train` with their labels, and `U_test` with the features of the 100 test points. The last lines check that the numbers are complete and unchanged, so that everybody works on the same data. They compute a hash, a short fingerprint of the numbers, and compare it with the one it must have.

In [ ]:
import hashlib

import numpy as np

U_train = np.array([
    [0.9764, 0.2052], [0.3966, 0.5951], [0.4196, 0.7450], [0.0149, 0.6328], [0.5373, 0.8528],
    [0.9452, 0.2172], [0.4587, 0.8132], [0.8747, 0.2427], [0.6255, 0.3612], [0.5842, 0.5614],
    [0.0512, 0.3466], [0.0158, 0.6218], [0.6689, 0.4065], [0.3897, 0.6721], [0.3816, 0.6786],
    [0.4389, 0.0329], [0.4215, 0.7700], [0.9543, 0.3684], [0.9515, 0.6963], [0.0132, 0.2910],
    [0.5730, 0.9146], [0.9262, 0.2315], [0.5104, 0.7003], [0.3140, 0.8553], [0.1348, 0.9972],
    [0.3931, 0.8536], [0.8288, 0.2113], [0.3547, 0.8530], [0.4614, 0.9119], [0.1172, 0.7471],
    [0.0709, 0.2352], [0.1426, 0.4401], [0.9713, 0.0655], [0.2329, 0.7412], [0.5305, 0.0215],
    [0.9261, 0.9651], [0.1940, 0.4600], [0.0291, 0.4596], [0.9499, 0.7343], [0.0963, 0.7138],
    [0.4121, 0.5500], [0.1841, 0.8515], [0.8134, 0.5030], [0.0947, 0.2489], [0.2943, 0.9119],
    [0.9123, 0.6453], [0.7431, 0.4363], [0.7086, 0.5343], [0.7273, 0.0851], [0.7454, 0.9475],
    [0.1616, 0.1187], [0.7086, 0.1648], [0.5137, 0.0373], [0.3596, 0.6734], [0.6922, 0.0856],
    [0.1564, 0.0109], [0.9144, 0.0275], [0.2351, 0.3582], [0.8209, 0.4147], [0.5573, 0.2588],
])
y_train = np.array([
    1, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 1, 1, 1, 0, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0,
    1, 1, 0, 1, 1, 0, 1, 0, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 0, 0, 0, 0,
])
U_test = np.array([
    [0.2483, 0.6469], [0.7026, 0.9986], [0.1699, 0.5478], [0.9218, 0.6455], [0.6452, 0.3035],
    [0.8654, 0.6677], [0.4204, 0.7589], [0.7080, 0.4304], [0.2770, 0.9303], [0.8332, 0.8496],
    [0.7950, 0.3280], [0.1268, 0.4283], [0.1766, 0.9228], [0.7056, 0.5138], [0.2476, 0.0034],
    [0.9098, 0.1738], [0.7432, 0.2641], [0.7487, 0.5193], [0.2398, 0.9408], [0.9317, 0.6472],
    [0.1936, 0.6844], [0.8215, 0.6527], [0.0290, 0.7162], [0.1171, 0.5588], [0.1865, 0.5856],
    [0.1714, 0.2404], [0.4760, 0.2391], [0.7505, 0.5244], [0.8571, 0.6707], [0.1510, 0.7103],
    [0.4681, 0.1645], [0.7470, 0.1399], [0.0635, 0.3154], [0.5365, 0.7981], [0.7207, 0.5539],
    [0.0141, 0.8265], [0.4658, 0.8459], [0.2354, 0.0743], [0.3552, 0.1393], [0.1683, 0.1403],
    [0.8673, 0.8750], [0.3463, 0.9204], [0.4573, 0.8666], [0.4065, 0.5538], [0.0932, 0.3513],
    [0.3445, 0.6133], [0.6995, 0.4805], [0.4963, 0.5952], [0.2586, 0.4845], [0.4228, 0.1482],
    [0.9872, 0.6376], [0.0608, 0.0064], [0.0482, 0.1348], [0.4424, 0.3894], [0.8057, 0.1525],
    [0.0033, 0.7621], [0.1444, 0.4531], [0.4137, 0.1858], [0.3987, 0.6041], [0.9787, 0.4648],
    [0.8640, 0.1530], [0.4616, 0.9864], [0.6723, 0.5882], [0.1424, 0.8732], [0.0758, 0.9511],
    [0.0151, 0.0588], [0.0203, 0.8268], [0.3452, 0.7342], [0.9855, 0.3288], [0.6244, 0.7085],
    [0.8284, 0.2277], [0.4970, 0.2598], [0.1799, 0.1152], [0.2069, 0.4724], [0.6718, 0.2009],
    [0.7368, 0.4486], [0.8731, 0.0547], [0.1995, 0.0665], [0.8033, 0.1491], [0.4334, 0.7264],
    [0.7784, 0.6037], [0.3738, 0.6957], [0.2554, 0.2313], [0.1453, 0.1349], [0.1694, 0.6057],
    [0.8846, 0.7980], [0.3386, 0.9011], [0.9995, 0.4136], [0.2924, 0.9040], [0.7472, 0.2748],
    [0.8812, 0.8203], [0.9494, 0.2936], [0.8371, 0.3524], [0.3530, 0.5330], [0.4398, 0.1813],
    [0.4348, 0.1923], [0.9210, 0.5854], [0.0713, 0.3419], [0.7319, 0.3857], [0.7185, 0.3059],
])

DATA_SHA256 = "b3ec0fadd3676621548b59e78e9f049d533b228367e1fe3297f093b8b9db255b"
fingerprint = hashlib.sha256(repr((U_train.tolist(), y_train.tolist(), U_test.tolist())).encode()).hexdigest()
assert fingerprint == DATA_SHA256, "The data in this cell was changed. Download the challenge notebook again."
print("training points:", len(U_train), "| labels 0 and 1 among them:", int((y_train == 0).sum()), "and", int((y_train == 1).sum()))
print("test points:", len(U_test), "| features from", U_test.min(), "to", U_test.max())

The second cell holds the feature map and the two functions of Thursday's Lab 2, unchanged, and imports the classifier of Lab 3. The lines that use them are under "Your attempt".

- `fmap = zz_feature_map(2, reps=2)` is the ZZ feature map for two features, with its block of gates twice.
- `state(x)` puts one point into the feature map and returns its feature state: the state of the two qubits that the feature map makes from the point. It comes as a state vector, the list of numbers that describes a state exactly. `x` holds two angles.
- `kernel(A, B)` returns a table of kernel values, the kernel matrix. The entry in row $i$ and column $j$ is the squared overlap of the feature states of point $i$ of `A` and point $j$ of `B`. The overlap of two states says how alike they are: squared, it is 1 for two equal states and 0 for two states that have nothing in common. This kernel has nothing to do with the Jupyter kernel that the Setup section mentions.
- `SVC` is scikit-learn's support vector classifier, a standard classifier that draws the border between two classes from such a table. With `kernel="precomputed"` it is trained on kernel values, not on features.

In [ ]:
from qiskit.circuit.library import zz_feature_map
from qiskit.quantum_info import Statevector
from sklearn.svm import SVC

fmap = zz_feature_map(2, reps=2)

def state(x):
    return Statevector(fmap.assign_parameters(x))

def kernel(A, B):
    sa = [state(x) for x in A]
    sb = [state(x) for x in B]
    return np.array([[abs(a.inner(b)) ** 2
                      for b in sb] for a in sa])

The third cell is new. `estimate(scale)` judges a setting from the training set alone, which is the only honest way when the test labels are hidden.

How it works: the 60 training points are split into five parts of 12. Each part in turn is set aside. The classifier is trained on the other 48 points and predicts the labels of the 12 that it has not seen. `estimate` returns the fraction of right predictions, averaged over the five parts. This procedure is called cross-validation. Its result is an estimate of the accuracy on new points. The five parts are fixed, so the same setting gives the same value every time. But the estimate rests on 60 points only, and it can be off by a tenth or more.

- `scale` is the number that every feature is multiplied by.
- `reps` and `C` are the two other settings that Thursday's notebook tried: the repetitions of the feature map, and how hard the classifier tries to get every training point right.

The cell ends with the estimate for Thursday's $\frac{\pi}{2}$.

In [ ]:
from sklearn.model_selection import StratifiedKFold, cross_val_score


def estimate(scale, reps=2, C=1.0):
    """Estimate the accuracy of a setting by five-fold cross-validation on the training set.

    scale: one number. Every feature is multiplied by it before it becomes an angle.
    reps: repetitions of the feature map, a whole number of 1 or more.
    C: the setting of the classifier, a number above 0.
    The five parts are fixed, so the same setting always gives the same value.
    """
    other_map = zz_feature_map(2, reps=reps)
    states = [Statevector(other_map.assign_parameters(scale * u)) for u in U_train]
    table = np.array([[abs(a.inner(b)) ** 2 for b in states] for a in states])
    parts = StratifiedKFold(5, shuffle=True, random_state=0)      # five parts, each with both classes
    return float(cross_val_score(SVC(kernel="precomputed", C=C), table, y_train, cv=parts).mean())


print("estimate for Thursday's pi/2:", round(estimate(np.pi / 2), 4))

The estimate for $\frac{\pi}{2}$ is 0.65. A classifier that guesses gets 0.5, so with this number the classifier has learned little. The estimate even flatters it: on the hidden test set this setting does no better than guessing, as "What to expect" said. With only 60 training points an estimate can be off by that much. Trust large differences between two estimates, not small ones.

The fourth cell checks the format of a set of predicted labels and writes the line you turn in.

- `problems(labels)` lists the reasons why a set of labels cannot be turned in. An empty list means that the format is right. It says nothing about whether the labels are correct. Nothing in this notebook can tell you that.
- `submission_line(labels)` writes the line: the 100 labels in the order of the test points. It stops with an error if `problems` finds something.

In [ ]:
def is_label(value):
    """True for the number 0 or 1, also written as 0.0 or 1.0 or as False or True."""
    return isinstance(value, (bool, int, float, np.integer, np.floating, np.bool_)) and value in (0, 1)


def problems(labels):
    """Return the list of reasons why these labels cannot be turned in. An empty list means the format is right."""
    try:
        if iter(labels) is labels:
            return ["labels must be a list or an array, not a generator or a map(...) object"]
        values = list(labels)
    except TypeError:
        return ["labels must be a list or an array with one label for each test point"]
    found = []
    if len(values) != len(U_test):
        found.append("labels needs exactly " + str(len(U_test)) + " entries, one for each test point")
    if not all(is_label(value) for value in values):
        found.append("every label must be the number 0 or 1, in a flat list or array with one entry for each test point")
    return found


def submission_line(labels):
    """The one line of text that you turn in. Stops with the reasons if problems(labels) finds any."""
    found = problems(labels)
    if found:
        raise ValueError("These labels cannot be turned in: " + "; ".join(found))
    return "QFF26-C4 labels=" + "".join(str(int(value)) for value in labels)

## Your attempt

From here on the work is yours.

The first cell below holds the number, `SCALE`. As given it is Thursday's $\frac{\pi}{2}$. The lines after it are Thursday's Labs 2 and 3 with one change: the features become angles by multiplying them by `SCALE`, where Thursday's notebook used a scaler, a function that stretched the features to the range from 0 to $\frac{\pi}{2}$. The cell trains the classifier on the 60 training points and predicts the 100 test labels. It also remembers, in `labels_scale`, the number that these labels were made with, and keeps a copy of the labels in `labels_made`.

What is left to you is the search for the number. `estimate` is the tool for it:

- One call of `estimate` takes a few hundredths of a second, so you can try hundreds of numbers. A loop over a range of numbers that keeps the best estimate is called a scan. A scan over several hundred numbers takes seconds, or a minute on a slow computer.
- The estimate can change a lot between two numbers that lie close together. Choose the step of a scan with that in mind.
- Because the labels were made with this feature map, the right number lets the classifier fit the training set very well. A number whose estimate is only a little better than its neighbors is probably not it.
- If several numbers in a row share the best estimate, take one from the middle of that run, not one at its end.
- `reps` and `C` are there if you want to try them. "What you are told about the data", in "The task in detail" above, says which `reps` made the labels.

Nobody tells you which of your labels were wrong, and only three lines count. Decide with the estimate, not with the form.

In [ ]:
SCALE = np.pi / 2          # Thursday's number. The number you find goes here.

X_train, X_test = SCALE * U_train, SCALE * U_test      # from features to angles

K_train = kernel(X_train, X_train)
K_test = kernel(X_test, X_train)

clf = SVC(kernel="precomputed")
clf.fit(K_train, y_train)

labels = clf.predict(K_test)          # the predicted label of every test point
labels_scale = SCALE                  # the number these labels were made with
labels_made = np.array(labels).copy() # a copy, so that the next cell notices if labels is changed later
print("estimate from the training set for this number:", round(estimate(SCALE), 4))
print("predicted labels 0 and 1:", int((labels == 0).sum()), "and", int((labels == 1).sum()))

The next cell checks the format of `labels` and prints your line. It cannot tell you whether a label is right. It does check that `SCALE` and `labels` are still the ones that the first cell of "Your attempt" used and made. If you set `SCALE` or `labels` somewhere else, for example in a search, put your number into the first line of that cell, where it says `SCALE = ...`, and run that cell again first. If your search also used another `C`, write it into the `SVC(...)` line of that cell, as `SVC(kernel="precomputed", C=...)`. Run unchanged, that cell sets `SCALE` back to Thursday's number.

In [ ]:
trouble = problems(labels)
if not np.isclose(SCALE, labels_scale):
    trouble.append("SCALE was changed after these labels were made. Put your number into the line SCALE = ... at the top of the first cell of Your attempt, and run that cell again")
elif not np.array_equal(np.asarray(labels), labels_made):
    trouble.append("labels was changed after the first cell of Your attempt made it. Put your number into the line SCALE = ... at the top of that cell, and run that cell again")
if trouble:
    print("These labels cannot be turned in: " + "; ".join(trouble))
else:
    if np.isclose(SCALE, np.pi / 2):
        print("SCALE is still Thursday's number. With the other settings as given, labels made with it score 0.")
    print("Your line:")
    print(submission_line(labels))
    print("Only your first three well-formed lines count. Send this one only when you are ready.")

## Turn it in

1. Copy the whole line that starts with `QFF26-C4`. Do not change anything in it.
2. Open the submission form: [https://forms.gle/9gY4TnmQqMiZ5gHM8](https://forms.gle/9gY4TnmQqMiZ5gHM8)
3. Enter the email you registered with and a leaderboard name, paste the line, and send. The name is shown on the leaderboards of the event page. Type the same name every time: the boards keep the name you typed first in any of the five forms, and only an organizer can change it.

Only your first three well-formed lines count, and the best of them is your score. A fourth line is ignored, and sending the same line again uses up one of the three. Nobody tells you which labels were wrong: the board on the event page shows only your points. So check your reasoning before you send a line, not after.

## If something breaks

| What you see | What to do |
|---|---|
| The setup cell stops and asks for a restart | Restart the kernel, then run the notebook again from the top. It installed the tested packages |
| "The data in this cell was changed" | A number in the data cell was edited or deleted. Download the challenge notebook again |
| The estimate stays below 0.8, and is sometimes below 0.5 | The number is not near the right one. Thursday's $\frac{\pi}{2}$ gives 0.65 here |
| A scan takes very long | Count the calls first: each one takes a few hundredths of a second, so all 901 numbers from 1.00 to 10.00 take about 15 seconds. A wide step can jump over the right number, so do not narrow the search from a coarse scan alone |
| `ValueError: Mismatching number of values and parameters` | `state(x)` needs a point with exactly two angles, and `estimate` needs `reps` of 1 or more |
| "labels needs exactly 100 entries" | `labels` must come from `clf.predict(K_test)`, with `K_test` made from all 100 test points |
| `NameError: name '...' is not defined`, for example for `np`, `kernel` or `estimate` | Run the cells under "The tools" first, from the top. For `SCALE`, `labels`, `labels_scale` or `labels_made`, also run the first cell of "Your attempt" |
| "SCALE was changed after these labels were made" or "labels was changed after the first cell of Your attempt made it" | Put your number into the first line of the first cell of "Your attempt", where it says `SCALE = ...`, and run that cell. Run unchanged, it sets `SCALE` back to Thursday's number |

## Where to go from here

- Every challenge stays open until Sun, Nov 22 at 11:59 PM, and a full score counts whenever it arrives before then. The first ten people to reach 100 also get a bonus (see the event page), but only three lines count, so send a line only when your estimate has convinced you.
- Stuck on the notebook itself? Ask in Discord between sessions (discord.gg/FBNzRcxMr), also if your score on the board looks wrong. Ask about the code, not about the number or the labels: the challenge is scored individually.